# 04 — Audit the feature relationships

This notebook explains **which feature relationships reject the candidates saved by notebook 03**. It checks the original 360 CTU rules separately, reproduces the saved constraint decisions, and repeats the calculations in float64 to inspect numerical effects.

**CPU runtime → Runtime → Run all → upload your notebook 03 results ZIP when prompted.** Use the full `diagnostics32_..._results_....zip`, not a screenshot or notebook. At the end, a new audit ZIP downloads automatically.

This reads your saved experiment. It does not rerun attacks, train a model, change candidates, or change the original acceptance threshold. Only the small metadata CSV is downloaded after setup; the full dataset and checkpoints are not needed. Earlier notebooks and result files are preserved.


## 1. Create a separate audit session

Keep the defaults. If you already placed the results ZIP in Colab, you can enter its full path in `INPUT_ZIP` to skip the upload dialog. Setup uses the same isolated CPU environment as the earlier experiments; a GPU is unnecessary.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

INPUT_ZIP = "" #@param {type:"string"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}
REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis_RelationshipAudit")).resolve()
HOST_PYTHON, HOST_VERSION = sys.executable, sys.version
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} is not a git checkout. Choose a new workspace.")
for required in ["scripts/bootstrap.py", "baseline/relationship_audit.py", "notebooks/04_Relationship_Audit.ipynb"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the updated repository.")

# Existing checkouts are preserved; no pull/reset/checkout is performed.
def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("relationship_audit")
SESSION.mkdir(parents=True, exist_ok=False)
source_paths = [p for folder in ["baseline", "scripts", "tests"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/04_Relationship_Audit.ipynb"]
context = {
    "project_repository": REPO_URL,
    "project_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
    "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    "experiment": "saved_candidate_relationship_audit", "evaluation_device": "cpu",
}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
for p in source_paths:
    target = SESSION / "source_snapshot" / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Export partial logs with the last cell.")

print("Project commit:", context["project_commit"])
print("New session:", SESSION)


## 2. Upload the notebook 03 ZIP

Choose **one complete results ZIP** from notebook 03. A copy is retained in this new session, so the audit export includes its source evidence. The runner reads arrays with pickling disabled and never executes scripts from the uploaded ZIP.


In [ ]:
input_folder = SESSION / unique_name("input")
input_folder.mkdir()
requested_path = os.environ.get("PHD_DIAGNOSTICS_ZIP", INPUT_ZIP).strip()
if requested_path:
    source_archive = Path(requested_path).expanduser().resolve()
    if not source_archive.is_file() or source_archive.suffix.lower() != ".zip":
        raise FileNotFoundError("INPUT_ZIP must point to the full notebook 03 results ZIP.")
    INPUT_ARCHIVE = input_folder / source_archive.name
    shutil.copy2(source_archive, INPUT_ARCHIVE)
else:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("Set INPUT_ZIP to your notebook 03 ZIP path outside Colab.") from exc
    previous_directory = Path.cwd()
    try:
        os.chdir(input_folder)
        uploaded = files.upload()
    finally:
        os.chdir(previous_directory)
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one notebook 03 results ZIP, then rerun this cell.")
    filename = next(iter(uploaded))
    INPUT_ARCHIVE = input_folder / Path(filename).name
    del uploaded
    if not INPUT_ARCHIVE.is_file() or INPUT_ARCHIVE.suffix.lower() != ".zip":
        raise ValueError("Select the full notebook 03 results ZIP.")
print("Source ZIP:", INPUT_ARCHIVE.name)
print("SHA-256:", hashlib.sha256(INPUT_ARCHIVE.read_bytes()).hexdigest())


## 3. Prepare the pinned rule engine

Uses the original Python 3.8.20 / NumPy 1.23.5 environment and pinned upstream source. Setup can take a few minutes on a fresh runtime. Colab's notebook interpreter stays unchanged.


In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
setup_copy = SESSION / unique_name("setup_report")
shutil.copy2(ROOT / ".cache/setup_report.json", setup_copy.with_suffix(".json"))
print("Pinned audit interpreter:", PYTHON)


## 4. Run the audit

For every saved stage, the audit recomputes all four constraint masks and checks them against notebook 03. It records each relationship's residual, affected original rows, failures within the attack budget, and concrete feature-value examples.

**Float64 is a diagnostic comparison.** It recalculates the stored float32 values; it cannot recover their earlier precision or turn an invalid candidate into a reported baseline success. The baseline tolerance remains zero.


In [ ]:
AUDIT = SESSION / unique_name("audit")
run_command([PYTHON, ROOT / "baseline/relationship_audit.py",
             "--repo", ROOT / "vendor/tabularbench",
             "--input-zip", INPUT_ARCHIVE, "--out", AUDIT], "audit.log")
print("Completed audit:", AUDIT)


## 5. Read the findings

`native_invalid` counts candidates rejected by the original relationship checker. `native_invalid_but_float64_valid` counts those whose **entire set** of relationship failures disappears in float64. Inspect individual-rule changes separately in `rule_summary.csv`.

The second table highlights rules that fail **within the fixed attack budget**. Counts overlap across rules, and multiple candidates can belong to the same original record. Bytes, packets, and duration residuals have different units.


In [ ]:
import pandas as pd
from IPython.display import display

report = json.loads((AUDIT / "audit_report.json").read_text())
summary = pd.read_csv(AUDIT / "stage_summary.csv")
rules = pd.read_csv(AUDIT / "rule_summary.csv")
display(summary[["generations", "model", "stage", "inputs", "candidates", "native_invalid",
                 "native_invalid_but_float64_valid", "native_valid_but_float64_invalid",
                 "rows_with_changed_valid_in_budget"]].sort_values(["generations", "model", "stage"]))
print("All saved constraint masks reproduced:", all(s["all_saved_constraint_masks_reproduced"] for s in report["stages"]))
print("Individual rule/candidate failures cleared in float64:", report["aggregate"]["failed_rule_pairs_cleared_in_float64"])
print("Individual rule/candidate failures newly exposed in float64:", report["aggregate"]["new_failed_rule_pairs_in_float64"])
print("CTU equality rules eligible for the upstream repair helper:",
      report["equality_rules_eligible_for_upstream_repair"], "of", report["equality_rules"])

highest_budget = rules["generations"].max()
top = rules[(rules["generations"] == highest_budget) & (rules["failing_candidates_in_budget"] > 0)]
top = top.sort_values(["model", "stage", "failing_candidates_in_budget", "failing_candidates"],
                      ascending=[True, True, False, False]).groupby(["model", "stage"], sort=False).head(5)
display(top[["model", "stage", "rule_id", "label", "units", "failing_candidates_in_budget",
             "failing_candidates", "native_residual_min", "native_residual_max"]])
print("Residual ranges in this table cover all stage candidates; the in-budget column gives the separate count.")
print("Original reported evasions (no new model inference):")
display(pd.DataFrame(report["reported_original_outcomes"])[["generations", "model", "reported_new_evasions"]])
print("Full evidence: rule_catalog.json, rule_summary.csv, family_summary.csv, stages/*/*/*/largest_residual_examples.json")


## 6. Download the audit evidence

The new ZIP contains the input ZIP, source snapshot, logs, environment record, rule catalog, summaries, candidate-level decisions, residual arrays, and feature-value examples. **Share this new ZIP.** You can run this cell separately after a failure to export partial logs.

Optional Google Drive copying requests Drive access only if you enable it above. Previous exports are preserved.


In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
archive = Path(shutil.make_archive(str(export_base), "zip", root_dir=SESSION.parent, base_dir=SESSION.name))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


## What this supports next

If failures persist and feature totals or ordering disagree, investigate a method that generates mutually consistent feature changes while preserving immutable features, bounds, integer counts and the same distance budget. Recheck every condition after any proposed repair.

The two CTU byte equalities compare sums; the pinned upstream repair helper selects only equalities with a single feature on the left. That explains why enabling its repair flags does not itself repair those two rules. This is an implementation limitation to investigate, not proof that a proposed replacement works or is novel.

Develop and tune future changes on a separate development partition. Keep the already inspected test subset as exploratory evidence; use a frozen evaluation protocol for later claims. These CTU records are aggregated features, not a live packet IDS experiment.
